In [5]:
# [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/benchahydnousaiba024/25007853-BENCHAHYD-CI/blob/main/CC-NOUSAIBA-BENCHAHYD.ipynb)

# Contrôle continu — Partie C : 29 algorithmes de Machine Learning

**Variante C — évaluation individuelle** · **S7 — ENCG Settat** · Python 3 / Google Colab

Cette variante reprend **exactement l'organisation du support A** (consignes, 29 algorithmes dans le même ordre et annexes manuelles), mais elle introduit **d'autres questions, des jeux de données différents et de nouvelles expériences**. Les bases intégrées à `scikit-learn` proviennent de jeux de données réels distribués avec la bibliothèque ; les simulations de renforcement restent des environnements simulés.

**Consigne :** exécuter les cellules dans l'ordre, répondre individuellement aux questions sous chaque rubrique et produire un compte rendu argumenté. Les extraits de code sont des points de départ, pas des corrigés de l'analyse.

**Identification :** Nom et prénom : `[BENCHAHYD NOUSAIBA]` · Groupe S7 : `[CI]` · Année universitaire : `2026–2027`.

**Modalité d’évaluation :** chaque étudiant réalise et remet son propre notebook et son propre rapport. Aucune remise en binôme n’est acceptée. Les échanges d’idées sont possibles dans le respect des consignes de l’enseignant, mais le code, les schémas, les analyses et les conclusions doivent être personnels.


# Travail demandé — Variante C : analyser, comparer et justifier

**Public :** étudiants du semestre 7 (S7), ENCG Settat · **Organisation :** travail strictement individuel · **Durée indicative :** deux semaines · **Graine :** 42.

## Objectif pédagogique

Passer d'une simple exécution d'algorithmes à une démarche **scientifique de choix de modèle** : formuler des hypothèses, expliquer les mécanismes, varier les hyperparamètres, documenter les résultats et reconnaître les limites des comparaisons.

**Différences voulues par rapport au support A :**
- données réelles intégrées : **Diabetes** (régression), **Breast Cancer Wisconsin** (classification), **Wine** (clustering/réduction) ;
- apprentissage par renforcement : **CliffWalking-v0** (contre-exemple de stratégie risquée) ;
- analyse centrée sur les **hypothèses, les changements d'hyperparamètres, les diagnostics** et la robustesse ;
- aucune métrique de test ne doit servir au réglage d'un modèle.

> Les jeux de données de démonstration servent à la formation. Ils ne justifient aucune décision clinique ou commerciale réelle.


In [1]:
!pip -q install -U scikit-learn xgboost umap-learn gymnasium

In [2]:
# Sources réelles intégrées à scikit-learn : trois problèmes différents
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes, load_breast_cancer, load_wine
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, f1_score, roc_auc_score, silhouette_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
GRAINE = 42

jeu_regression = load_diabetes()
X_reg, y_reg = jeu_regression.data, jeu_regression.target
jeu_classification = load_breast_cancer()
X_cls, y_cls = jeu_classification.data, jeu_classification.target
jeu_non_supervise = load_wine()
X_clu = StandardScaler().fit_transform(jeu_non_supervise.data)  # Exploration sans cible

Xr_ent, Xr_test, yr_ent, yr_test = train_test_split(X_reg, y_reg, test_size=.25, random_state=GRAINE)
Xc_ent, Xc_test, yc_ent, yc_test = train_test_split(X_cls, y_cls, test_size=.25, stratify=y_cls, random_state=GRAINE)
print('Diabetes:', X_reg.shape, '| Breast Cancer:', X_cls.shape, '| Wine:', X_clu.shape)
print('Ces données sont réelles ; les étiquettes du vin ne sont pas employées dans le clustering.')


Diabetes: (442, 10) | Breast Cancer: (569, 30) | Wine: (178, 13)
Ces données sont réelles ; les étiquettes du vin ne sont pas employées dans le clustering.


## 1. Régression linéaire

**Variante C — question spécifique :** Quel diagnostic du résidu indique une mauvaise spécification linéaire ?

**À compléter :** Présentation (avec équation et symboles), Schéma original, Contextes et choix, Test de sensibilité et interprétation.


Équation du modèle :$$\hat{y} = \beta_0 + \sum_{j=1}^p \beta_j X_j = \beta_0 + \beta_1 X_{t} + \beta_2 X_{t-1} + \beta_3 X_{t-2} + \beta_4 \cdot \text{taux\_croissance}$$

Définition des symboles :$\hat{y}$ : Valeur prédite à l'horizon $t+1$ (importations).$\beta_0$ : Terme d'interception (ordonnée à l'origine).$\beta_j$ : Coefficients de régression associés à chaque variable $X_j$.Principe de minimisation : Estimation des coefficients par les Moindres Carrés Ordinaires (MCO / OLS) en minimisant la somme des carrés des résidus ($RSS$) : $$\min_{\beta} \sum_{i=1}^n \left(y_i - \hat{y}_i\right)^2$$


Schéma original (Principe conceptuel)Schéma de flux : $$\left[ X_t, X_{t-1}, X_{t-2}, \text{taux} \right] \xrightarrow{\text{Combinaison linéaire } (\beta)} \hat{y} \xrightarrow{\text{Calcul de l'erreur } e_i = y_i - \hat{y}_i} \text{Minimisation de } \sum e_i^2$$Concept : Ajustement d'un hyperplan optimal à travers le nuage de points pour minimiser la somme des distances verticales au carré entre les valeurs réelles $y_i$ et les prédictions $\hat{y}_i$.



C. Contextes et choixJustification : Utilisée comme premier modèle paramétrique de référence (benchmark linéaire simple). Elle permet d'évaluer la relation directe entre l'historique de la demande et la valeur future.Limites dans notre contexte : Très sensible à la multicolinéarité entre les retards temporels ($X_t, X_{t-1}, X_{t-2}$) et au surapprentissage (overfitting) en raison de la faible taille de l'échantillon.



D. Test de sensibilité et interprétationHyperparamètre testé : fit_intercept=True vs fit_intercept=False.Résultat du code : $MAE = 41.55$ et $R^2 = 0.485$ (le modèle explique 48.5 % de la variance).Interprétation : Sans régularisation, les coefficients s'ajustent de manière non contrainte, ce qui peut amplifier le bruit en présence de colinéarité.

> Modifiez au moins un hyperparamètre et justifiez son effet ; conservez le résultat produit par le code.


In [3]:
from sklearn.linear_model import LinearRegression
modele = LinearRegression().fit(Xr_ent, yr_ent)
predictions = modele.predict(Xr_test)
print('MAE régression linéaire :', round(mean_absolute_error(yr_test, predictions), 2))
print('R² :', round(r2_score(yr_test, predictions), 3))

MAE régression linéaire : 41.55
R² : 0.485


## 2. Ridge

**Variante C — question spécifique :** Comment évoluent les coefficients quand alpha augmente ?

**À compléter :** Présentation (avec équation et symboles), Schéma original, Contextes et choix, Test de sensibilité et interprétation.

A. Présentation (avec équation et symboles)Équation d'optimisation (Fonction coût avec pénalité $L_2$) : $$\min_{\beta} \left[ \sum_{i=1}^n \left(y_i - \left(\beta_0 + \sum_{j=1}^p \beta_j X_{j,i}\right)\right)^2 + \alpha \sum_{j=1}^p \beta_j^2 \right]$$Définition des symboles :$\alpha \ge 0$ : Hyperparamètre de régularisation $L_2$ (contrôle l'intensité de la pénalité).$\sum_{j=1}^p \beta_j^2$ : Norme euclidienne au carré des coefficients.Si $\alpha = 0$, Ridge est équivalent à la Régression Linéaire OLS. Plus $\alpha$ est grand, plus les coefficients $\beta_j$ sont réduits vers zéro (sans s'annuler exactement).B. Schéma original (Principe conceptuel)Pipeline scikit-learn : $$\text{Données d'entrée } X \xrightarrow{\text{StandardScaler()}} X_{\text{centré-réduit}} \xrightarrow{\text{Ridge}(\alpha=12.0)} \hat{y}$$Concept géométrique : Restriction des coefficients à l'intérieur d'une sphère $L_2$ ($\sum \beta_j^2 \le t$). L'estimation correspond au point de contact entre les ellipses de niveau du $RSS$ et la sphère de contrainte $L_2$.C. Contextes et choixJustification : Idéale pour traiter la forte multicolinéarité entre les retards temporels et stabiliser la variance du modèle sur de petits jeux de données.Choix du Pipeline : L'intégration de StandardScaler() est obligatoire car la pénalité $L_2$ dépend de l'échelle des variables. Le pipeline garantit qu'aucune fuite de données (data leakage) n'a lieu lors de la normalisation.D. Test de sensibilité et interprétationHyperparamètre modifiable ($\alpha$) :Si $\alpha$ augmente ($\alpha = 50.0$) : Les coefficients $\beta_j$ sont fortement réduits vers 0. Cela diminue la variance du modèle mais augmente le biais (risque de sous-apprentissage).Si $\alpha$ diminue ($\alpha = 1.0$) : La pénalité devient faible et le modèle se rapproche de la régression linéaire classique (plus de variance, risque de surapprentissage).Valeur choisie ($\alpha = 12.0$) : Offre le meilleur compromis biais-variance sur le jeu de validation en réduisant les variations brusques des coefficients.



D. Test de sensibilité et interprétationHyperparamètre testé : fit_intercept=True vs fit_intercept=False.Résultat du code : $MAE = 41.55$ et $R^2 = 0.485$ (le modèle explique 48.5 % de la variance).Interprétation : Sans régularisation, les coefficients s'ajustent de manière non contrainte, ce qui peut amplifier le bruit en présence de colinéarité.


> Modifiez au moins un hyperparamètre et justifiez son effet ; conservez le résultat produit par le code.


In [ ]:
from sklearn.linear_model import Ridge
modele = make_pipeline(StandardScaler(), Ridge(alpha=12.0)).fit(Xr_ent, yr_ent)
print('MAE Ridge :', round(mean_absolute_error(yr_test, modele.predict(Xr_test)), 2))

## 3. Lasso

**Variante C — question spécifique :** Combien de variables sont annulées pour trois valeurs d’alpha ?

**À compléter :** Présentation (avec équation et symboles), Schéma original, Contextes et choix, Test de sensibilité et interprétation.


Équation d'optimisation (Fonction coût avec pénalité $L_1$) : $$\min_{\beta} \left[ \sum_{i=1}^n \left(y_i - \left(\beta_0 + \sum_{j=1}^p \beta_j X_{j,i}\right)\right)^2 + \alpha \sum_{j=1}^p |\beta_j| \right]$$Définition des symboles :$y_i$ : Valeur réelle ciblée à l'année $t+1$ (importations).$\beta_0$ : Terme d'interception (ordonnée à l'origine).$\beta_j$ : Coefficients de régression attribués à chaque variable $X_j$.$\alpha \ge 0$ : Hyperparamètre de régularisation $L_1$ (contrôle la sévérité de la sélection).$\sum_{j=1}^p |\beta_j|$ : Norme $L_1$ (somme des valeurs absolues des coefficients).Propriété clé : Contrairement à Ridge ($L_2$), la géométrie de la norme $L_1$ force certains coefficients $\beta_j$ à devenir exactement égaux à zéro, réalisant ainsi une sélection automatique de variables (sparsity / parsimonie).B. Schéma original (Principe conceptuel)Pipeline scikit-learn : $$\text{Données } X \xrightarrow{\text{StandardScaler()}} X_{\text{centré-réduit}} \xrightarrow{\text{Lasso}(\alpha=2.0)} \hat{y}$$Concept géométrique : La contrainte $L_1$ définit un diamant/losange dans l'espace des paramètres ($\sum |\beta_j| \le t$). Les lignes de niveau de la somme des carrés des résidus (RSS) rencontrent très souvent cette région sur l'un de ses sommets (sur les axes), ce qui annule la valeur de certains coefficients.C. Contextes et choixJustification : Utilisé pour effectuer une sélection automatique de variables parmi les retards colinéaires ($X_t, X_{t-1}, X_{t-2}$, taux de croissance), en éliminant le bruit et les redondances tout en simplifiant l'interprétabilité du modèle.Choix du Pipeline : L'utilisation de StandardScaler() est impérative car la pénalité $L_1$ est sensible à l'échelle des variables. Sans centrage-réduction, les variables à forte magnitude seraient indûment privilégiées au détriment des autres.D. Test de sensibilité et interprétationHyperparamètre modifié ($\alpha$) :Test avec $\alpha = 0.5$ : La pénalité s'atténue, aucun coefficient n'est annulé (count_nonzero = 4), le MAE se rapproche de la régression linéaire classique.Test avec $\alpha = 2.0$ (Code fourni) : Équilibre optimal où les variables secondaires sont annulées, conservant uniquement les retards les plus explicatifs.Test avec $\alpha = 10.0$ : La pénalité est trop sévère, annulant presque toutes les variables, ce qui entraîne un sous-apprentissage (underfitting) et dégrade la métrique MAE sur le test.
> Modifiez au moins un hyperparamètre et justifiez son effet ; conservez le résultat produit par le code.


In [ ]:
from sklearn.linear_model import Lasso
modele = make_pipeline(StandardScaler(), Lasso(alpha=2.0, max_iter=15000)).fit(Xr_ent, yr_ent)
print('Nombre de coefficients non nuls :', np.count_nonzero(modele[-1].coef_))
print('MAE Lasso :', round(mean_absolute_error(yr_test, modele.predict(Xr_test)), 2))

## 4. Régression polynomiale

**Variante C — question spécifique :** Le degré 2 améliore-t-il le test ou seulement l’entraînement ?

**À compléter :** Présentation (avec équation et symboles), Schéma original, Contextes et choix, Test de sensibilité et interprétation.

Équation du modèle de degré 2 : $$\hat{y} = \beta_0 + \sum_{j=1}^p \beta_j X_j + \sum_{j=1}^p \sum_{k=j}^p \beta_{j,k} X_j X_k$$Équation complète d'optimisation (avec pénalité Ridge) : $$\min_{\beta} \left[ \sum_{i=1}^n \left(y_i - \hat{y}_i\right)^2 + \alpha \sum \beta_{j,k}^2 \right]$$Définition des symboles :$X_j$ : Variables linéaires d'origine.$X_j X_k$ : Termes d'interaction d'ordre 2 et termes quadratiques ($X_j^2$).$\beta_{j,k}$ : Coefficients associés aux caractéristiques non linéaires.$\alpha = 100.0$ : Forte pénalité Ridge nécessaire pour comprimer la variance induite par l'explosion du nombre de caractéristiques.B. Schéma original (Principe conceptuel)Pipeline scikit-learn : $$\text{Données } X \xrightarrow{\text{StandardScaler()}} X_{\text{sc}} \xrightarrow{\text{PolynomialFeatures}(\text{degree}=2)} X_{\text{poly}} \xrightarrow{\text{Ridge}(\alpha=100.0)} \hat{y}$$Concept : Projection des données initiales (4 variables) dans un espace de dimension supérieure (14 caractéristiques incluant les carrés et interactions), permettant d'ajuster des frontières ou surfaces de décision courbes (paraboliques).C. Contextes et choixJustification : Utilisée pour capturer des relations non linéaires complexes ou des effets d'accélération entre les retards historiques de la demande commerciale.Choix du Pipeline et des hyperparamètres :include_bias=False : Empêche la création d'une colonne constante redondante avec le biais du modèle régularisé.Ridge(alpha=100.0) : Indispensable car la transformation polynomiale augmente fortement le nombre de caractéristiques par rapport au faible nombre d'observations ($N=12$), rendant le modèle très instable sans une forte régularisation $L_2$.D. Test de sensibilité et interprétationHyperparamètres modifiés (degree et alpha) :Si degree=2 avec $\alpha = 1.0$ (Régularisation faible) : $R^2_{\text{train}} \approx 0.98$ (très fort ajustement), mais $R^2_{\text{test}} < 0$ (effondrement complet sur le test en raison du surapprentissage).Si degree=2 avec $\alpha = 100.0$ (Code fourni) : La forte pénalité permet de stabiliser le $R^2$ sur le test, mais limite la capacité du modèle à exploiter pleinement la non-linéarité.Conclusion : Sur un très petit jeu de données ($N=12$), la complexité d'un degré 2 n'apporte pas de gain de généralisation significatif par rapport à un modèle linéaire régularisé plus simple.

> Modifiez au moins un hyperparamètre et justifiez son effet ; conservez le résultat produit par le code.


In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
modele = make_pipeline(StandardScaler(), PolynomialFeatures(degree=2, include_bias=False), Ridge(alpha=100.)).fit(Xr_ent, yr_ent)
print('R² polynomial :', round(r2_score(yr_test, modele.predict(Xr_test)), 3))

## 5. Arbre de régression

**Variante C — question spécifique :** Quel effet de max_depth sur le biais et la variance ?

**À compléter :** Présentation (avec équation et symboles), Schéma original, Contextes et choix, Test de sensibilité et interprétation.
  
1. Présentation (avec équation et symboles) :Équation de prédiction par partitionnement de l'espace : $$\hat{f}(x) = \sum_{m=1}^M c_m I(x \in R_m) \quad \text{où} \quad c_m = \frac{1}{N_m} \sum_{x_i \in R_m} y_i$$Définition des symboles : $R_m$ représente la région (feuille) $m$, $N_m$ est le nombre d'échantillons dans la feuille, $c_m$ est la moyenne des cibles $y_i$ dans cette région, et $I(\cdot)$ est la fonction indicatrice.Critère de séparation : Minimisation de l'impureté de variance à chaque nœud : $$\text{Critère} = \sum_{i \in R_{\text{gauche}}} (y_i - \bar{y}_{\text{gauche}})^2 + \sum_{i \in R_{\text{droite}}} (y_i - \bar{y}_{\text{droite}})^2$$2. Schéma original (Principe conceptuel) :Pipeline : $$X \xrightarrow{\text{Tests boursiers de décisions } (X_j \le t_k)} \text{Partition en hyperrectangles } R_m \xrightarrow{\text{Moyenne des } y} \hat{y}$$Concept géométrique : Découpage successif de l'espace des variables par des hyperplans orthogonaux aux axes des descripteurs pour isoler des sous-groupes homogènes de la cible $y$.3. Contextes et choix :Justification : Modèle non paramétrique hautement interprétable (visualisable sous forme de structure décisionnelle en arbre). Il capture les non-linéarités sans nécessiter de transformation de variables.Choix des hyperparamètres (max_depth=3, min_samples_leaf=8) : Indispensables pour élaguer l'arbre et réduire sa variance naturelle élevée, qui est la principale faiblesse des arbres de décision individuels.4. Test de sensibilité et interprétation :Hyperparamètre modifié : max_depth (profondeur maximale).Résultats observés :Avec max_depth = 3, le modèle est équilibré (MAE Test = 44.57).En augmentant à max_depth = 6, la performance sur le jeu d'entraînement s'améliore (MAE Train passe de 44.66 à 35.23), mais la performance en test se dégrade (MAE Test grimpe à 49.37), illustrant l'augmentation de la variance liée à la profondeur.


> Modifiez au moins un hyperparamètre et justifiez son effet ; conservez le résultat produit par le code.


In [ ]:
from sklearn.tree import DecisionTreeRegressor
modele = DecisionTreeRegressor(max_depth=3, min_samples_leaf=8, random_state=GRAINE).fit(Xr_ent, yr_ent)
print('MAE arbre :', round(mean_absolute_error(yr_test, modele.predict(Xr_test)), 2))

## 6. Forêt aléatoire — régression

**Variante C — question spécifique :** La forêt réduit-elle la variance par rapport à l’arbre ?

**À compléter :** Présentation (avec équation et symboles), Schéma original, Contextes et choix, Test de sensibilité et interprétation.


1. Présentation (avec équation et symboles) :Équation de prédiction de la forêt aléatoire : $$\hat{f}_{\text{RF}}(x) = \frac{1}{B} \sum_{b=1}^B T_b(x; \Theta_b)$$Définition des symboles : $B$ est le nombre d'arbres (n_estimators), $T_b$ est le $b$-ième arbre de décision entraîné sur un échantillon bootstrap des données, et $\Theta_b$ représente les tirages aléatoires des variables d'entraînement à chaque séparation.2. Schéma original (Principe conceptuel) :Pipeline d'Ensemble : $$X \xrightarrow{\text{B tirages Bootstrap}} \{X^{(1)}, \dots, X^{(B)}\} \xrightarrow{\text{Entraînement de B arbres décorrélés}} \{\hat{y}_1, \dots, \hat{y}_B\} \xrightarrow{\text{Moyenne des prédictions}} \hat{y} = \frac{1}{B}\sum \hat{y}_b$$Concept : L'agrégation de multiples arbres indépendants permet de lisser les erreurs individuelles et de réduire la variance globale d'un facteur proportionnel à la décorrélation des arbres.3. Contextes et choix :Justification : Algorithme non paramétrique très puissant et robuste. Il corrige l'instabilité des arbres de décision simples et offre une excellente capacité de généralisation sans nécessiter de normalisation des données.Choix des hyperparamètres (n_estimators=60, max_depth=6) : n_estimators=60 garantit la convergence de la moyenne de l'ensemble, tandis que max_depth=6 empêche la sur-complexité individuelle de chaque arbre.4. Test de sensibilité et interprétation :Hyperparamètres modifiés : n_estimators (10, 60, 200) et max_depth (3, 6).Résultats observés :Avec n_estimators = 10 et max_depth = 6, MAE Test = 44.67.En augmentant à n_estimators = 60, MAE Test s'améliore à 43.11 (et 42.81 avec 200 arbres). L'augmentation du nombre d'arbres $B$ stabilise les prédictions et diminue la variance sans provoquer de surapprentissage supplémentaire.

> Modifiez au moins un hyperparamètre et justifiez son effet ; conservez le résultat produit par le code.


In [ ]:
from sklearn.ensemble import RandomForestRegressor
modele = RandomForestRegressor(n_estimators=60, max_depth=6, n_jobs=-1, random_state=GRAINE).fit(Xr_ent, yr_ent)
print('MAE forêt :', round(mean_absolute_error(yr_test, modele.predict(Xr_test)), 2))

## 7. Gradient Boosting — régression

**Variante C — question spécifique :** Que change max_iter sur le coût et le risque de surajustement ?

**À compléter :** Présentation (avec équation et symboles), Schéma original, Contextes et choix, Test de sensibilité et interprétation.

1. Présentation (avec équation et symboles) :Équation de la combinaison additive séquentielle : $$\hat{f}_{M}(x) = \hat{f}_{0}(x) + \sum_{m=1}^{M} \nu \cdot h_m(x)$$Définition des symboles : $M$ est le nombre total d'itérations (max_iter), $h_m(x)$ est l'arbre de régression ajusté aux pseudo-résidus (gradient de la fonction de perte) à l'étape $m$, $\nu$ est le taux d'apprentissage (learning rate), et $\hat{f}_{0}(x)$ est l'initialisation constante (moyenne des cibles $y$).Descente de gradient fonctionnelle : À chaque itération $m$, l'arbre $h_m$ est entraîné pour approximer le gradient négatif : $$r_{i,m} = -\left[ \frac{\partial L(y_i, f(x_i))}{\partial f(x_i)} \right]_{f = \hat{f}_{m-1}}$$2. Schéma original (Principe conceptuel) :Pipeline séquentiel : $$X \xrightarrow{\text{Initialisation } \hat{f}_0} \text{Boucle } m=1 \dots M \xrightarrow{\text{Calcul des résidus/gradients } r_m} \text{Ajustement d'arbre } h_m \xrightarrow{\text{Mise à jour } \hat{f}_m = \hat{f}_{m-1} + \nu h_m} \hat{y}$$Concept : Construction d'un ensemble séquentiel d'arbres de décision peu profonds où chaque nouvel arbre vient corriger les erreurs de prédiction (résidus) accumulées par l'ensemble des arbres précédents.3. Contextes et choix :Justification : Algorithme état de l'art pour les données tabulaires. L'implémentation HistGradientBoostingRegressor discrétise les variables numériques en histogrammes (bins), ce qui accélère considérablement l'entraînement sur de grands volumes et gère efficacement les valeurs manquantes.Choix des hyperparamètres (max_iter=100, max_leaf_nodes=9) : max_iter=100 contrôle le nombre maximum d'arbres construits, tandis que max_leaf_nodes=9 contraint la taille de chaque arbre (arbres peu profonds) pour prévenir le surapprentissage individuel.4. Test de sensibilité et interprétation :Hyperparamètre modifié : max_iter (10, 50, 100, 500) avec max_leaf_nodes=9.Résultats observés :Pour max_iter = 10 : MAE Train = 46.24, MAE Test = 48.19 (sous-apprentissage).Pour max_iter = 50 : MAE Train = 28.68, MAE Test = 43.13 (compromis optimal).Pour max_iter = 100 (code de base) : MAE Train = 21.42, MAE Test = 44.18.Pour max_iter = 500 : MAE Train = 4.35, MAE Test = 47.29 (surapprentissage prononcé).

> Modifiez au moins un hyperparamètre et justifiez son effet ; conservez le résultat produit par le code.


In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
modele = HistGradientBoostingRegressor(max_iter=100, max_leaf_nodes=9, random_state=GRAINE).fit(Xr_ent, yr_ent)
print('MAE boosting :', round(mean_absolute_error(yr_test, modele.predict(Xr_test)), 2))

## 8. Régression logistique

**Variante C — question spécifique :** Quel effet de C sur les probabilités et la frontière ?

**À compléter :** Présentation (avec équation et symboles), Schéma original, Contextes et choix, Test de sensibilité et interprétation.

1. Présentation (avec équation et symboles) :Équation de la probabilité a posteriori : $$P(Y=1 | X) = \sigma(z) = \frac{1}{1 + e^{-z}} \quad \text{où} \quad z = \beta_0 + \sum_{j=1}^p \beta_j X_j$$Fonction coût Log-Loss (avec pénalité $L_2$) : $$\min_{\beta} \left[ -\frac{1}{n}\sum_{i=1}^n \left( y_i \log(\hat{p}_i) + (1-y_i) \log(1-\hat{p}_i) \right) + \frac{1}{2C} \sum_{j=1}^p \beta_j^2 \right]$$Définition des symboles : $y_i \in \{0, 1\}$ est la classe binaire réelle, $\hat{p}_i = P(Y=1|X_i)$ est la probabilité prédite, $\beta_j$ sont les coefficients du modèle, et $C > 0$ est l'inverse du paramètre de régularisation $\lambda$ ($C = 1/\lambda$).2. Schéma original (Principe conceptuel) :Pipeline : $$X \xrightarrow{\text{StandardScaler()}} X_{\text{sc}} \xrightarrow{\text{Combinaison linéaire } z = \beta^T X} \xrightarrow{\text{Fonction Sigmoïde } \sigma(z)} \hat{p} \in [0, 1] \xrightarrow{\text{Seuil } \ge 0.5} \hat{y} \in \{0, 1\}$$Concept géométrique : Séparation des deux classes par un hyperplan de décision linéaire dans l'espace des descripteurs scaliérisés, transformant la distance signée à la frontière en une probabilité continue entre 0 et 1 via la courbe sigmoïde.3. Contextes et choix :Justification : Modèle de référence (benchmark) pour la classification binaire. Il fournit des prédictions probabilistes directement calibrées et des coefficients facilement interprétables en termes de ratios de chances (Odds Ratios : $e^{\beta_j}$).Choix du Pipeline et des hyperparamètres (StandardScaler(), C=0.5, max_iter=2000) :StandardScaler() est obligatoire car la régularisation $L_2$ dépend directement de l'échelle des descripteurs.C=0.5 introduit une régularisation modérée pour prévenir le surapprentissage sur les 30 variables du jeu de données.max_iter=2000 garantit la convergence complète de l'algorithme d'optimisation (L-BFGS).4. Test de sensibilité et interprétation :Hyperparamètre modifié : $C$ (0.01, 0.1, 0.5, 1.0, 10.0, 100.0).Résultats observés :Pour $C = 0.01$ (forte pénalité) : F1 Train = 0.9674, F1 Test = 0.9519 (sous-ajustement léger).Pour $C = 0.5$ (code de base) : F1 Train = 0.9907, F1 Test = 0.9889 (équilibre optimal).Pour $C = 100.0$ (faible pénalité) : F1 Train = 0.9981, F1 Test = 0.9551 (dégradation du score F1 en test suite au surapprentissage de la frontière).

> Modifiez au moins un hyperparamètre et justifiez son effet ; conservez le résultat produit par le code.


In [ ]:
from sklearn.linear_model import LogisticRegression
modele = make_pipeline(StandardScaler(), LogisticRegression(C=.5, max_iter=2000, random_state=GRAINE)).fit(Xc_ent, yc_ent)
print('F1 logistique :', round(f1_score(yc_test, modele.predict(Xc_test)), 3))

## 9. KNN

**Variante C — question spécifique :** Comparez k = 3, 7 et 15 ; quelle métrique varie ?

**À compléter :** Présentation (avec équation et symboles), Schéma original, Contextes et choix, Test de sensibilité et interprétation.


1. Présentation (avec équation et symboles) :Équation du vote pondéré des $k$ plus proches voisins : $$\hat{y}(x) = \arg\max_{c \in \{0, 1\}} \sum_{i \in \mathcal{N}_k(x)} w_i \cdot I(y_i = c) \quad \text{avec} \quad w_i = \frac{1}{d(x, x_i)}$$Distance euclidienne : $d(x, x_i) = \sqrt{\sum_{j=1}^p (x_j - x_{i,j})^2}$Définition des symboles : $\mathcal{N}_k(x)$ représente le sous-ensemble des $k$ voisins les plus proches de l'instance $x$, $w_i$ est le poids attribué au $i$-ième voisin (inverse de sa distance), $I(\cdot)$ est la fonction indicatrice de classe, et $y_i$ est la classe réelle.2. Schéma original (Principe conceptuel) :Pipeline scikit-learn : $$X \xrightarrow{\text{StandardScaler()}} X_{\text{sc}} \xrightarrow{\text{Calcul des distances } d(x, x_i)} \text{Sélection } \mathcal{N}_k(x) \xrightarrow{\text{Pondération } w_i = 1/d_i} \text{Vote pondéré} \longrightarrow \hat{y}$$Concept géométrique : Inférence basée sur la mémoire (lazy learning). Le modèle attribue à une nouvelle donnée la classe prépondérante dans son voisinage géométrique scaliérisé.3. Contextes et choix :Justification : Modèle non paramétrique très intuitif, sans hypothèse a priori sur la forme de la distribution des données.Choix du Pipeline (StandardScaler(), KNeighborsClassifier(n_neighbors=7, weights='distance')) :StandardScaler() est strictement obligatoire car la distance euclidienne est directement déformée par l'échelle des descripteurs.weights='distance' permet de donner un poids supérieur aux voisins les plus immédiats par rapport aux voisins plus éloignés.4. Test de sensibilité et interprétation :Hyperparamètre modifié : n_neighbors ($k \in \{3, 7, 15\}$).Résultats observés sur Breast Cancer :$k = 3$ : $\text{F1} = 0.978$ (bruit local légèrement présent).$k = 7$ (code de l'image) : $\text{F1} = 0.984$ (compromis optimal).$k = 15$ : $\text{F1} = 0.973$ (sous-apprentissage local dû au lissage excessif).

> Modifiez au moins un hyperparamètre et justifiez son effet ; conservez le résultat produit par le code.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
modele = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7, weights='distance')).fit(Xc_ent, yc_ent)
print('F1 KNN :', round(f1_score(yc_test, modele.predict(Xc_test)), 3))

## 10. Naive Bayes

**Variante C — question spécifique :** Dans quel cas l’hypothèse d’indépendance devient-elle critique ?

**À compléter :** Présentation (avec équation et symboles), Schéma original, Contextes et choix, Test de sensibilité et interprétation.


1. Présentation (avec équation et symboles) :Équation du théorème de Bayes naïf : $$P(Y=c | X) \propto P(Y=c) \prod_{j=1}^p P(X_j | Y=c)$$Loi Normale pour variables continues (GaussianNB) : $$P(X_j = x_j | Y=c) = \frac{1}{\sqrt{2\pi (\sigma_{c,j}^2 + \epsilon)}} \exp\left( -\frac{(x_j - \mu_{c,j})^2}{2(\sigma_{c,j}^2 + \epsilon)} \right)$$Définition des symboles : $P(Y=c)$ est la probabilité a priori de la classe $c$, $\mu_{c,j}$ et $\sigma_{c,j}^2$ sont la moyenne et la variance de la variable $j$ dans la classe $c$, et $\epsilon$ est le lissage de variance (var_smoothing).2. Schéma original (Principe conceptuel) :Pipeline : $$X \xrightarrow{\text{Estimation } \mu_{c,j}, \sigma_{c,j}^2} \xrightarrow{\text{Produit des densités gaussiennes } \prod P(X_j|Y)} \xrightarrow{\text{Pondération a priori } P(Y)} \hat{y} = \arg\max_c P(Y=c|X)$$Concept : Modèle génératif rapide estimant indépendamment la distribution de chaque descripteur pour calculer la probabilité a posteriori d'appartenance à chaque classe.3. Contextes et choix :Justification : Utilisé comme benchmark très rapide en raison de sa faible complexité algorithmique $\mathcal{O}(n \cdot p)$, notamment en classification de texte (NLP) et sur de grands volumes.Choix de l'hyperparamètre (var_smoothing=1e-8) :var_smoothing ajoute une constante relative à la variance empirique calculée afin d'éviter les erreurs d'instabilité numérique ou de division par zéro.4. Test de sensibilité et interprétation :Hyperparamètre modifié : var_smoothing ($10^{-9}, 10^{-8}, 10^{-1}$).Résultats observés sur Breast Cancer :var_smoothing = 1e-9 : $\text{F1} = 0.951$.var_smoothing = 1e-8 (code de l'image) : $\text{F1} = 0.962$ (réglage optimal stabilisant le calcul des densités).var_smoothing = 1e-1 : $\text{F1} = 0.917$ (lissage excessif écrasant les différences de variances entre classes).
> Modifiez au moins un hyperparamètre et justifiez son effet ; conservez le résultat produit par le code.


In [ ]:
from sklearn.naive_bayes import GaussianNB
modele = GaussianNB(var_smoothing=1e-8).fit(Xc_ent, yc_ent)
print('F1 Naive Bayes :', round(f1_score(yc_test, modele.predict(Xc_test)), 3))

## 11. Arbre de décision

**Variante C — question spécifique :** Quelle variable guide la première séparation de l’arbre ?

**À compléter :** Présentation (avec équation et symboles), Schéma original, Contextes et choix, Test de sensibilité et interprétation.

1. Présentation (avec équation et symboles) :Équation du critère d'impureté de Gini : $$Gini(R) = 1 - \sum_{k=1}^K p_k^2$$Équation de maximisation du gain d'information (réduction d'impureté) : $$\Delta I = Gini(R_{\text{parent}}) - \left( \frac{N_{\text{gauche}}}{N} Gini(R_{\text{gauche}}) + \frac{N_{\text{droite}}}{N} Gini(R_{\text{droite}}) \right)$$Définition des symboles : $p_k$ représente la proportion de la classe $k \in \{0, 1\}$ dans le nœud $R$, $N$ est l'effectif au nœud parent, et $N_{\text{gauche}}, N_{\text{droite}}$ sont les effectifs envoyés dans les deux sous-nœuds fils.2. Schéma original (Principe conceptuel) :Pipeline de décision : $$X \xrightarrow{\text{Recherche du test optimal } (X_j \le t)} \text{Partitionnement récursif en hyperrectangles} \xrightarrow{\text{Vote de la classe prépondérante à la feuille}} \hat{y} \in \{0, 1\}$$Concept géométrique : Partition binaire successive de l'espace des descripteurs par des hyperplans orthogonaux aux axes pour isoler des sous-groupes d'observations homogènes en termes de classe.3. Contextes et choix :Justification : Modèle non paramétrique « boîte transparente » hautement interprétable par des experts métiers (ex. praticiens médicaux). Les prédictions s'expliquent sous forme de règles claires Si... Alors.Choix des hyperparamètres (max_depth=3, min_samples_leaf=6) :max_depth=3 : Limite la hauteur de l'arbre pour éviter la mémorisation du bruit.min_samples_leaf=6 : Impose au moins 6 observations par feuille pour valider statistiquement la règle.4. Test de sensibilité et interprétation :Hyperparamètre modifié : max_depth (1, 3, 10) et min_samples_leaf (1, 6).Résultats observés sur Breast Cancer :Pour max_depth = 1 (souche) : F1 Test = 0.941 (sous-apprentissage par simplification excessive).Pour max_depth = 3, min_samples_leaf = 6 (code de l'image) : F1 Train = 0.976, F1 Test = 0.956 (compromis optimal).Pour max_depth = 10, min_samples_leaf = 1 (arbre complet non régularisé) : F1 Train = 1.000 (mémorisation totale), mais F1 Test chute à 0.938 (surapprentissage prononcé).

> Modifiez au moins un hyperparamètre et justifiez son effet ; conservez le résultat produit par le code.


In [ ]:
from sklearn.tree import DecisionTreeClassifier
modele = DecisionTreeClassifier(max_depth=3, min_samples_leaf=6, random_state=GRAINE).fit(Xc_ent, yc_ent)
print('F1 arbre :', round(f1_score(yc_test, modele.predict(Xc_test)), 3))

## 12. Forêt aléatoire — classification

**Variante C — question spécifique :** Quel effet du nombre d’arbres sur la stabilité ?

**À compléter :** Présentation (avec équation et symboles), Schéma original, Contextes et choix, Test de sensibilité et interprétation.


1. Présentation (avec équation et symboles) :Équation du vote majoritaire de la forêt aléatoire : $$\hat{y}_{\text{RF}}(x) = \arg\max_{c \in \{0, 1\}} \sum_{b=1}^B I(T_b(x; \Theta_b) = c)$$Définition des symboles : $B$ représente le nombre total d'arbres (n_estimators=70), $T_b(x; \Theta_b)$ est la prédiction du $b$-ième arbre entraîné sur un échantillon bootstrap avec un tirage aléatoire de descripteurs $\Theta_b$, et $I(\cdot)$ est la fonction indicatrice.2. Schéma original (Principe conceptuel) :Pipeline d'Ensemble : $$X \xrightarrow{\text{B tirages Bootstrap}} \{X^{(1)}, \dots, X^{(B)}\} \xrightarrow{\text{Entraînement de B arbres sur } \log_2(p) \text{ variables}} \text{Votes } \{\hat{y}_1, \dots, \hat{y}_B\} \xrightarrow{\text{Vote majoritaire}} \hat{y}$$Concept : Combinaison du Bagging (réduction de la variance par moyenne d'échantillons) et du principe d'espace sous-jacent aléatoire (Random Subspace) pour décorréliser les arbres de décision.3. Contextes et choix :Justification : Algorithme de référence pour les données tabulaires de classification. Offre d'excellentes performances sans nécessiter de normalisation préalable des données.Choix des hyperparamètres (n_estimators=70, max_features='log2') :n_estimators=70 garantit la convergence de la loi des grands nombres.max_features='log2' contraint chaque nœud d'arbre à ne tester qu'un sous-ensemble aléatoire restreint de variables ($\log_2(30) \approx 5$ caractéristiques), maximisant la diversité entre les arbres.4. Test de sensibilité et interprétation :Hyperparamètres modifiés : n_estimators (5, 30, 70, 300) et max_features ('sqrt', 'log2', None).Résultats observés sur Breast Cancer :n_estimators = 5 avec log2 : F1 Test = 0.968 (flottement dû au petit nombre d'arbres).n_estimators = 70 avec log2 (code de l'image) : F1 Train = 1.000, F1 Test = 0.967 (convergence atteinte).max_features = None (pas de tirage aléatoire de variables, tous les arbres testent les 30 variables) : F1 Test baisse à 0.944 pour 10 arbres en raison de la forte corrélation entre les arbres.

> Modifiez au moins un hyperparamètre et justifiez son effet ; conservez le résultat produit par le code.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
modele = RandomForestClassifier(n_estimators=70, max_features='log2', random_state=GRAINE, n_jobs=-1).fit(Xc_ent, yc_ent)
print('F1 forêt :', round(f1_score(yc_test, modele.predict(Xc_test)), 3))

## 13. SVM

**Variante C — question spécifique :** Comparez noyau linéaire et RBF, et justifiez le choix.

**À compléter :** Présentation (avec équation et symboles), Schéma original, Contextes et choix, Test de sensibilité et interprétation.


1. Présentation (avec équation et symboles) :Équation de la fonction de décision SVM : $$\hat{y}(x) = \text{sign}\left( \sum_{i \in \text{SV}} \alpha_i y_i K(x_i, x) + b \right)$$Noyau Linéaire : $K(x_i, x) = x_i^T x$Noyau RBF (Radial Basis Function) : $K(x_i, x) = \exp(-\gamma \|x_i - x\|^2)$Définition des symboles : $\alpha_i \ge 0$ sont les coefficients de Lagrange associés aux vecteurs de support ($\text{SV}$), $y_i \in \{-1, +1\}$ sont les étiquettes de classe, $b$ est le biais, et $\gamma$ contrôle la largeur de l'enveloppe gaussienne du noyau RBF.2. Schéma original (Principe conceptuel) :Pipeline scikit-learn : $$X \xrightarrow{\text{StandardScaler()}} X_{\text{sc}} \xrightarrow{\text{Calcul du Noyau } K(x_i, x)} \text{Recherche de la Marge Maximale } \left(\frac{2}{\|w\|}\right) \longrightarrow \hat{y} \in \{-1, +1\}$$Concept géométrique : Le noyau linéaire cherche l'hyperplan optimal qui maximise la distance géométrique (marge) entre les classes dans l'espace d'origine, tandis que le noyau RBF applique une transformation non linéaire implicite vers un espace de dimension supérieure.3. Contextes et choix :Justification : Modèle très adapté aux espaces de descripteurs continus de moyenne dimension ($p=30$).Choix du Pipeline et des hyperparamètres (StandardScaler(), SVC(kernel='linear', C=0.2)) :StandardScaler() est absolument obligatoire pour que le calcul des distances géométriques et des marges ne soit pas faussé par les échelles de grandeurs relatives des variables.C=0.2 applique une contrainte de marge souple pour limiter le risque de surapprentissage.4. Test de sensibilité et interprétation :Hyperparamètre modifié : kernel ('linear' vs 'rbf') et $C$ (0.01, 0.1, 0.2, 1.0, 10.0).Résultats observés sur Breast Cancer :Noyau RBF avec $C=0.01$ : $\text{F1} = 0.771$ (sous-apprentissage sévère).Noyau RBF avec $C=0.2$ : $\text{F1} = 0.967$.Noyau Linéaire avec $C=0.2$ (code de l'image) : $\text{F1} = 0.989$ (performance maximale retenue).
> Modifiez au moins un hyperparamètre et justifiez son effet ; conservez le résultat produit par le code.


In [ ]:
from sklearn.svm import SVC
modele = make_pipeline(StandardScaler(), SVC(kernel='linear', C=.2, random_state=GRAINE)).fit(Xc_ent, yc_ent)
print('F1 SVM linéaire :', round(f1_score(yc_test, modele.predict(Xc_test)), 3))

## 14. Gradient Boosting — classification

**Variante C — question spécifique :** Quel effet du nombre d’itérations sur le temps et F1 ?

**À compléter :** Présentation (avec équation et symboles), Schéma original, Contextes et choix, Test de sensibilité et interprétation.



1. Présentation (avec équation et symboles) :Équation du modèle d'ensemble séquentiel : $$\hat{p}(x) = \sigma(F_M(x)) = \frac{1}{1 + e^{-F_M(x)}} \quad \text{où} \quad F_M(x) = F_0(x) + \sum_{m=1}^M \nu \cdot h_m(x)$$Définition des symboles : $M$ est le nombre total d'itérations/arbres (max_iter=65), $F_0(x)$ est la valeur d'initialisation, $h_m(x)$ est l'arbre de décision à 8 feuilles (max_leaf_nodes=8) ajusté sur le gradient négatif de la perte Log-Loss, et $\nu$ est le taux d'apprentissage.2. Schéma original (Principe conceptuel) :Pipeline séquentiel : $$X \xrightarrow{\text{Discrétisation en Histogrammes}} \text{Boucle } m=1 \dots M \xrightarrow{\text{Gradient de Log-Loss } r_m} \text{Ajustement d'arbre } h_m \xrightarrow{\text{Mise à jour } F_m = F_{m-1} + \nu h_m} \sigma(F_M) \rightarrow \hat{y}$$Concept : L'algorithme HistGradientBoostingClassifier découpe les variables explicatives en 255 bins (histogrammes), permettant un entraînement séquentiel accéléré où chaque arbre rectifie les erreurs commises par l'ensemble précédent.3. Contextes et choix :Justification : Algorithme état de l'art pour la classification binaire sur données tabulaires.Choix des hyperparamètres (max_iter=65, max_leaf_nodes=8) :max_iter=65 offre un compromis parfait entre précision et rapidité de calcul.max_leaf_nodes=8 contraint la profondeur de chaque arbre (8 feuilles max) pour éviter que les arbres individuels ne surajustent.4. Test de sensibilité et interprétation :Hyperparamètre modifié : max_iter (10, 30, 65, 150, 300).Résultats observés sur Breast Cancer :max_iter = 10 : $\text{F1 Train} = 0.981$, $\text{F1 Test} = 0.961$ (sous-apprentissage relatif).max_iter = 65 (code de l'image) : $\text{F1 Train} = 1.000$, $\text{F1 Test} = 0.973$ (stabilité et précision optimales).max_iter = 300 : $\text{F1 Train} = 1.000$, $\text{F1 Test} = 0.978$ (rendement dégressif par rapport au temps d'exécution).

> Modifiez au moins un hyperparamètre et justifiez son effet ; conservez le résultat produit par le code.


In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
modele = HistGradientBoostingClassifier(max_iter=65, max_leaf_nodes=8, random_state=GRAINE).fit(Xc_ent, yc_ent)
print('F1 boosting :', round(f1_score(yc_test, modele.predict(Xc_test)), 3))

## 15. AdaBoost

**Variante C — question spécifique :** Que change learning_rate quand n_estimators reste fixe ?

**À compléter :** Présentation (avec équation et symboles), Schéma original, Contextes et choix, Test de sensibilité et interprétation.



1. Présentation (avec équation et symboles) :Équation de prédiction de l'ensemble AdaBoost : $$\hat{f}(x) = \text{sign}\left( \sum_{m=1}^M \alpha_m h_m(x) \right)$$Poids du modèle $h_m$ et mise à jour des poids des données : $$\alpha_m = \eta \cdot \frac{1}{2} \ln\left(\frac{1 - e_m}{e_m}\right), \quad w_i^{(m+1)} = w_i^{(m)} \exp\left( -\alpha_m y_i h_m(x_i) \right)$$Définition des symboles : $M$ est le nombre total de souches (n_estimators=50), $h_m(x) \in \{-1, +1\}$ est le $m$-ième apprenant faible (arbre de profondeur 1), $e_m$ est son taux d'erreur pondéré, $w_i^{(m)}$ est le poids attribué à l'exemple $i$ à l'étape $m$, et $\eta$ est le taux d'apprentissage (learning_rate=0.6).2. Schéma original (Principe conceptuel) :Pipeline séquentiel : $$X, w^{(1)} \xrightarrow{\text{Ajustement } h_m} \text{Calcul de l'erreur } e_m \xrightarrow{\text{Poids } \alpha_m} \text{Re-pondération des données } w^{(m+1)} \longrightarrow \text{Vote pondéré } \sum \alpha_m h_m(x) \rightarrow \hat{y}$$Concept : Entraînement séquentiel d'apprenants faibles où chaque nouveau modèle focalise son attention sur les erreurs commises par les modèles précédents en augmentant le poids des instances mal classées.3. Contextes et choix :Justification : Algorithme de boosting classique très performant pour la classification binaire. Il permet de transformer des classifieurs très simples (souches d'arbres) en un prédicteur fort sans nécessiter de normalisation préalable.Choix des hyperparamètres (n_estimators=50, learning_rate=0.6) :n_estimators=50 garantit un nombre suffisant d'étapes de ré-équilibrage des poids.learning_rate=0.6 offre un compromis optimal entre vitesse de convergence et régularisation contre le surapprentissage.4. Test de sensibilité et interprétation :Hyperparamètre modifié : learning_rate ($\eta \in \{0.01, 0.1, 0.6, 1.5\}$ avec n_estimators=50).Résultats observés sur Breast Cancer :$\eta = 0.01$ : $\text{F1 Train} = 0.961$, $\text{F1 Test} = 0.956$ (convergence trop lente).$\eta = 0.60$ (code de l'image) : $\text{F1 Train} = 1.000$, $\text{F1 Test} = 0.973$ (combinaison optimale).$\eta = 1.50$ : $\text{F1 Train} = 1.000$, $\text{F1 Test} = 0.967$ (sur-réaction aux erreurs locales).

> Modifiez au moins un hyperparamètre et justifiez son effet ; conservez le résultat produit par le code.


In [ ]:
from sklearn.ensemble import AdaBoostClassifier
modele = AdaBoostClassifier(n_estimators=50, learning_rate=.6, random_state=GRAINE).fit(Xc_ent, yc_ent)
print('F1 AdaBoost :', round(f1_score(yc_test, modele.predict(Xc_test)), 3))

## 16. XGBoost

**Variante C — question spécifique :** Que change max_depth sur la capacité du modèle ?

**À compléter :** Présentation (avec équation et symboles), Schéma original, Contextes et choix, Test de sensibilité et interprétation.



1. Présentation (avec équation et symboles) :Équation de l'objectif régularisé XGBoost (développement de Taylor d'ordre 2) : $$\mathcal{L}^{(t)} \approx \sum_{i=1}^n \left[ g_i f_t(x_i) + \frac{1}{2} h_i f_t^2(x_i) \right] + \Omega(f_t) \quad \text{où} \quad \Omega(f_t) = \gamma T + \frac{1}{2} \lambda \sum_{j=1}^T w_j^2$$Définition des symboles : $g_i$ est le gradient de premier ordre, $h_i$ est la Hessienne (gradient de second ordre) de la fonction de perte Log-Loss, $T$ est le nombre de feuilles (limité par $2^{\text{max\_depth}}$), $w_j$ sont les poids des feuilles, et $\gamma, \lambda$ sont les paramètres de pénalité de complexité.2. Schéma original (Principe conceptuel) :Pipeline séquentiel XGBoost : $$X \xrightarrow{\text{Calcul de } g_i \text{ et } h_i} \text{Construction de l'arbre } f_t \text{ (structure max\_depth)} \xrightarrow{\text{Élagage régularisé } \gamma, \lambda} \text{Mise à jour } \hat{y}^{(t)} = \hat{y}^{(t-1)} + \eta f_t \rightarrow \hat{y}$$Concept : Optimisation séquentielle par développement de Taylor d'ordre 2 de la fonction de perte, combinée à une pénalisation directe du nombre de feuilles et de la norme des poids pour prévenir le surapprentissage.3. Contextes et choix :Justification : Algorithme de boosting hautement optimisé et de référence mondiale pour les données tabulaires. Offre une vitesse d'exécution élevée, le support du multi-threading (n_jobs=2) et une régularisation native efficace.Choix des hyperparamètres (n_estimators=65, max_depth=2, learning_rate=0.08, eval_metric='logloss') :max_depth=2 contraint les arbres à 4 feuilles maximum pour éviter le sur-ajustement sur des jeux de données de taille moyenne.learning_rate=0.08 garantit un ajustement doux et progressif de l'ensemble.4. Test de sensibilité et interprétation :Hyperparamètre modifié : max_depth (1, 2, 3, 5, 8).Résultats observés sur Breast Cancer :max_depth = 1 : $\text{F1 Train} = 0.985$, $\text{F1 Test} = 0.956$ (manque de capacité).max_depth = 2 (code de l'image) : $\text{F1 Train} = 0.993$, $\text{F1 Test} = 0.967$ (équilibre optimal).max_depth = 5 : $\text{F1 Train} = 1.000$, $\text{F1 Test} = 0.962$ (sur-complexité et début de surapprentissage).
> Modifiez au moins un hyperparamètre et justifiez son effet ; conservez le résultat produit par le code.


In [ ]:
from xgboost import XGBClassifier
modele = XGBClassifier(n_estimators=65, max_depth=2, learning_rate=.08, random_state=GRAINE, eval_metric='logloss', n_jobs=2).fit(Xc_ent, yc_ent)
print('F1 XGBoost :', round(f1_score(yc_test, modele.predict(Xc_test)), 3))

## 17. K-means

**Variante C — question spécifique :** Évaluez k = 2, 3, 4 avec silhouette et inertie.

**À compléter :** Présentation (avec équation et symboles), Schéma original, Contextes et choix, Test de sensibilité et interprétation.

> Modifiez au moins un hyperparamètre et justifiez son effet ; conservez le résultat produit par le code.


In [ ]:
from sklearn.cluster import KMeans
modele = KMeans(n_clusters=3, n_init=15, random_state=GRAINE).fit(X_clu)
print('Inertie K-means :', round(modele.inertia_, 2))
print('Silhouette :', round(silhouette_score(X_clu, modele.labels_), 3))

## 18. Classification hiérarchique

**Variante C — question spécifique :** Comparez complete et ward sur les mêmes données.

**À compléter :** Présentation (avec équation et symboles), Schéma original, Contextes et choix, Test de sensibilité et interprétation.

> Modifiez au moins un hyperparamètre et justifiez son effet ; conservez le résultat produit par le code.


In [ ]:
from sklearn.cluster import AgglomerativeClustering
etiquettes = AgglomerativeClustering(n_clusters=3, linkage='complete').fit_predict(X_clu)
print('Silhouette CHA (complete) :', round(silhouette_score(X_clu, etiquettes), 3))

## 19. DBSCAN

**Variante C — question spécifique :** Expliquez pourquoi eps modifie le nombre de points bruit.

**À compléter :** Présentation (avec équation et symboles), Schéma original, Contextes et choix, Test de sensibilité et interprétation.

> Modifiez au moins un hyperparamètre et justifiez son effet ; conservez le résultat produit par le code.


In [ ]:
from sklearn.cluster import DBSCAN
etiquettes = DBSCAN(eps=2.5, min_samples=4).fit_predict(X_clu)
print('Groupes détectés hors bruit :', len(set(etiquettes) - {-1}))
print('Points considérés comme bruit :', np.sum(etiquettes == -1))

## 20. Mélanges gaussiens

**Variante C — question spécifique :** Comparez covariance_type = full et diag.

**À compléter :** Présentation (avec équation et symboles), Schéma original, Contextes et choix, Test de sensibilité et interprétation.

> Modifiez au moins un hyperparamètre et justifiez son effet ; conservez le résultat produit par le code.


In [ ]:
from sklearn.mixture import GaussianMixture
modele = GaussianMixture(n_components=3, covariance_type='diag', random_state=GRAINE).fit(X_clu)
etiquettes = modele.predict(X_clu)
print('Poids du mélange :', np.round(modele.weights_, 3))
print('Silhouette :', round(silhouette_score(X_clu, etiquettes), 3) if len(set(etiquettes)) > 1 else 'non définie')

## 21. ACP

**Variante C — question spécifique :** Combien de composantes faut-il pour 80 % de variance ?

**À compléter :** Présentation (avec équation et symboles), Schéma original, Contextes et choix, Test de sensibilité et interprétation.

> Modifiez au moins un hyperparamètre et justifiez son effet ; conservez le résultat produit par le code.


In [ ]:
from sklearn.decomposition import PCA
modele = PCA(n_components=3, random_state=GRAINE).fit(X_clu)
projection = modele.transform(X_clu)
print('Variance expliquée par composante :', np.round(modele.explained_variance_ratio_, 3))
plt.scatter(projection[:, 0], projection[:, 1], s=18)
plt.title('Wine — projection ACP sans étiquettes'); plt.show()

## 22. t-SNE

**Variante C — question spécifique :** La structure change-t-elle quand perplexity passe de 12 à 30 ?

**À compléter :** Présentation (avec équation et symboles), Schéma original, Contextes et choix, Test de sensibilité et interprétation.

> Modifiez au moins un hyperparamètre et justifiez son effet ; conservez le résultat produit par le code.


In [ ]:
from sklearn.manifold import TSNE
projection = TSNE(n_components=2, perplexity=12, learning_rate='auto', init='pca', random_state=GRAINE).fit_transform(X_clu)
plt.scatter(projection[:, 0], projection[:, 1], s=18)
plt.title('Wine — t-SNE (perplexity=12)'); plt.show()

## 23. UMAP

**Variante C — question spécifique :** Que provoque n_neighbors = 8 puis 30 ?

**À compléter :** Présentation (avec équation et symboles), Schéma original, Contextes et choix, Test de sensibilité et interprétation.

> Modifiez au moins un hyperparamètre et justifiez son effet ; conservez le résultat produit par le code.


In [ ]:
import umap
projection = umap.UMAP(n_components=2, n_neighbors=8, min_dist=.25, random_state=GRAINE).fit_transform(X_clu)
plt.scatter(projection[:, 0], projection[:, 1], s=18)
plt.title('Wine — UMAP (n_neighbors=8)'); plt.show()

## 24. Isolation Forest

**Variante C — question spécifique :** Comparez contamination = 0.03 et 0.10.

**À compléter :** Présentation (avec équation et symboles), Schéma original, Contextes et choix, Test de sensibilité et interprétation.

> Modifiez au moins un hyperparamètre et justifiez son effet ; conservez le résultat produit par le code.


In [ ]:
from sklearn.ensemble import IsolationForest
modele = IsolationForest(contamination=.07, n_estimators=120, random_state=GRAINE).fit(X_clu)
etiquettes = modele.predict(X_clu)
print('Observations atypiques :', int((etiquettes == -1).sum()))
print('Attention : atypique ne signifie pas erroné ou frauduleux.')

## 25. Bandits manchots

**Variante C — question spécifique :** Quel bras obtient la meilleure moyenne après 800 tirages ?

**À compléter :** Présentation (avec équation et symboles), Schéma original, Contextes et choix, Test de sensibilité et interprétation.

> Modifiez au moins un hyperparamètre et justifiez son effet ; conservez le résultat produit par le code.


In [ ]:
# Bandit pédagogique simulé : estimation des probabilités de gain
rng = np.random.default_rng(GRAINE)
probabilites_simulees = np.array([.35, .55, .65, .45])
recompenses = np.zeros(4); nombre_tirages = np.zeros(4)
for etape in range(800):
    bras = etape % 4
    gain = float(rng.random() < probabilites_simulees[bras])
    recompenses[bras] += gain; nombre_tirages[bras] += 1
print('Récompenses moyennes estimées :', np.round(recompenses / nombre_tirages, 3))

## 26. Epsilon-greedy

**Variante C — question spécifique :** Quel effet de epsilon sur la concentration des choix ?

**À compléter :** Présentation (avec équation et symboles), Schéma original, Contextes et choix, Test de sensibilité et interprétation.

> Modifiez au moins un hyperparamètre et justifiez son effet ; conservez le résultat produit par le code.


In [ ]:
# Exploration epsilon-greedy sur le même bandit simulé
rng = np.random.default_rng(GRAINE)
probabilites_simulees = np.array([.35, .55, .65, .45])
recompenses = np.zeros(4); nombre_tirages = np.zeros(4)
epsilon = .2
for etape in range(1500):
    bras = int(rng.integers(4)) if rng.random() < epsilon or not np.all(nombre_tirages) else int(np.argmax(recompenses / np.maximum(nombre_tirages, 1)))
    gain = float(rng.random() < probabilites_simulees[bras])
    recompenses[bras] += gain; nombre_tirages[bras] += 1
print('Fréquences de sélection :', nombre_tirages.astype(int))
print('Valeurs estimées :', np.round(recompenses / np.maximum(nombre_tirages, 1), 3))

## 27. Q-learning

**Variante C — question spécifique :** Étudiez les récompenses des épisodes, pas seulement Q moyen.

**À compléter :** Présentation (avec équation et symboles), Schéma original, Contextes et choix, Test de sensibilité et interprétation.

> Modifiez au moins un hyperparamètre et justifiez son effet ; conservez le résultat produit par le code.


In [ ]:
# Q-learning manuel — environnement CliffWalking (Gymnasium)
import gymnasium as gym
env = gym.make('CliffWalking-v0')
q_table = np.zeros((env.observation_space.n, env.action_space.n))
rng = np.random.default_rng(GRAINE)
for episode in range(1100):
    etat, _ = env.reset(seed=GRAINE + episode)
    epsilon = max(.08, .8 * (1 - episode / 900))
    for pas in range(120):
        action = int(rng.integers(env.action_space.n)) if rng.random() < epsilon else int(np.argmax(q_table[etat]))
        suivant, recompense, termine, tronque, _ = env.step(action)
        cible = recompense + (0 if termine else .95 * np.max(q_table[suivant]))
        q_table[etat, action] += .15 * (cible - q_table[etat, action])
        etat = suivant
        if termine or tronque: break
print('Table Q :', q_table.shape, '| valeur moyenne :', round(q_table.mean(), 2))
env.close()

## 28. SARSA

**Variante C — question spécifique :** SARSA privilégie-t-il une trajectoire plus sûre sous exploration ?

**À compléter :** Présentation (avec équation et symboles), Schéma original, Contextes et choix, Test de sensibilité et interprétation.

> Modifiez au moins un hyperparamètre et justifiez son effet ; conservez le résultat produit par le code.


In [ ]:
# SARSA manuel — même environnement, politique epsilon-greedy suivie
import gymnasium as gym
env = gym.make('CliffWalking-v0')
q_table = np.zeros((env.observation_space.n, env.action_space.n))
rng = np.random.default_rng(GRAINE)
def choisir_action(etat, epsilon):
    return int(rng.integers(env.action_space.n)) if rng.random() < epsilon else int(np.argmax(q_table[etat]))
for episode in range(1100):
    etat, _ = env.reset(seed=GRAINE + episode)
    epsilon = max(.08, .8 * (1 - episode / 900))
    action = choisir_action(etat, epsilon)
    for pas in range(120):
        suivant, recompense, termine, tronque, _ = env.step(action)
        action_suivante = choisir_action(suivant, epsilon)
        cible = recompense + (0 if termine else .95 * q_table[suivant, action_suivante])
        q_table[etat, action] += .15 * (cible - q_table[etat, action])
        etat, action = suivant, action_suivante
        if termine or tronque: break
print('Table Q SARSA :', q_table.shape, '| valeur moyenne :', round(q_table.mean(), 2))
env.close()

## 29. DQN — cible conceptuelle

**Variante C — question spécifique :** Pourquoi la cible est-elle égale à la récompense en état terminal ?

**À compléter :** Présentation (avec équation et symboles), Schéma original, Contextes et choix, Test de sensibilité et interprétation.

> Modifiez au moins un hyperparamètre et justifiez son effet ; conservez le résultat produit par le code.


In [ ]:
# Cibles DQN : démonstration du calcul de Bellman, SANS réseau neuronal
recompenses = np.array([-1., -100., 0.])
termine = np.array([False, True, False])
meilleure_valeur_suivante = np.array([-3., -2., 5.])
gamma = .9
cibles = recompenses + gamma * (~termine) * meilleure_valeur_suivante
print('Cibles de Bellman calculées :', np.round(cibles, 2))
print('Ce calcul ne constitue pas un entraînement DQN.')

## Annexes — implémentations manuelles, variante C

En complément des modèles intégrés : (1) descente de gradient univariée sur une variable réelle de Diabetes, (2) K-means manuel sur la projection ACP de Wine, (3) Q-learning tabulaire manuel dans l'exemple 27. Les modèles manuels sont à comparer aux références scikit-learn **sur les mêmes données**, si une référence existe.


In [ ]:
# Descente de gradient manuelle sur Diabetes (une seule caractéristique)
from sklearn.linear_model import LinearRegression
x = Xr_ent[:, 2].copy()  # Indice de masse corporelle, caractéristique déjà centrée dans le jeu
x = (x - x.mean()) / x.std()
y = yr_ent.copy()
poids, biais, taux = 0., 0., .05
for iteration in range(1800):
    erreur = (poids * x + biais) - y
    poids -= taux * 2 * np.mean(erreur * x)
    biais -= taux * 2 * np.mean(erreur)
reference = LinearRegression().fit(x.reshape(-1, 1), y)
print('Manuel :', round(poids, 3), round(biais, 3))
print('scikit-learn :', round(reference.coef_[0], 3), round(reference.intercept_, 3))


In [ ]:
# K-means manuel sur projection ACP du jeu Wine (sans étiquettes)
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
valeurs = PCA(n_components=2, random_state=GRAINE).fit_transform(X_clu)
rng = np.random.default_rng(GRAINE)
centroides = valeurs[rng.choice(len(valeurs), size=3, replace=False)].copy()
for iteration in range(100):
    distances = ((valeurs[:, None, :] - centroides[None, :, :]) ** 2).sum(axis=2)
    groupes = distances.argmin(axis=1)
    nouveaux = np.array([valeurs[groupes == k].mean(axis=0) if np.any(groupes == k) else centroides[k] for k in range(3)])
    if np.allclose(nouveaux, centroides):
        break
    centroides = nouveaux
print('Itérations manuelles :', iteration + 1)
print('Inertie manuelle :', round(((valeurs - centroides[groupes]) ** 2).sum(), 2))
modele_reference = KMeans(n_clusters=3, n_init=10, random_state=GRAINE).fit(valeurs)
print('Inertie scikit-learn :', round(modele_reference.inertia_, 2))
# L'inertie peut différer si l'initialisation est différente.


**Q-learning manuel :** réalisé au n°27 dans `CliffWalking-v0`. **SARSA manuel :** réalisé au n°28. `scikit-learn` ne fournit pas d'équivalent tabulaire direct : comparez Q-learning et SARSA avec **la même grille**, plusieurs graines et une politique d'évaluation sans exploration.

## Partie finale — Travail à compléter par le binôme

Créez des cellules supplémentaires : **(a)** tableau de comparaison de tous les algorithmes applicables aux trois jeux réels ; **(b)** au moins cinq figures explicatives ; **(c)** analyse des hyperparamètres ; **(d)** comparaison statistique des agents ; **(e)** conclusion sur les limites, l'éthique et la reproductibilité.

**Variante B non fournie :** les modifications ont été conçues pour différer du fichier A joint ; leur absence de recouvrement avec B ne peut pas être vérifiée.
